In [ ]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as ticker
from scipy.integrate import odeint
import util
from scipy.stats import nbinom, gamma

### Figure 1

In [ ]:
def I0Ii_model(y, t, beta, k):
    dydt = np.zeros(k)
    dydt[0] = -beta * y[0] 
    for i in range(1, k-1):
        dydt[i] = -beta * y[i] + beta * y[i - 1]  
    dydt[k-1] = beta * y[k-2] 
    return dydt

nu = 1  # Rate of progression through compartments
y0 = np.zeros(41)
y0[0] = 1  # Initial number of I0
t = np.linspace(0, 40, 4000)

solution = odeint(I0Ii_model, y0, t, args=(nu, 41))
compartments = [solution[:, i] for i in range(41)]
plt.figure(figsize=(12, 8))
for i in range(41-1):
    plt.plot(t, compartments[i], label=f'I{i}')
plt.xlabel('Time (days)')
plt.ylabel('Probability')
plt.title('Probability Distribution Across Each Compartment Over Time')
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
n = 40
c_i = util.generate_neg_binom(n, 11.528428093645484, 0.8) # Case detection profile
omega_i = util.generate_neg_binom(n, 3.1515151515151514, 0.24444444444444444) # Shedding load profile
f_i= util.generate_neg_binom(n, 3.8989898989898992, 0.4473684210526315) # Infectivity profile 

compartment_probs_over_time = solution[:,:n]
shedding_concentration_over_time = np.dot(compartment_probs_over_time, omega_i)
fi_over_time = np.dot(compartment_probs_over_time, f_i)
ci_over_time = np.dot(compartment_probs_over_time, c_i)

In [ ]:
np.random.seed(48)
mean_sld = 6.7
sd_sld = 7.0
shape_sld = (mean_sld / sd_sld) ** 2
scale_sld = (sd_sld ** 2) / mean_sld

mean_incubation = 5.3
sd_incubation = 3.2
shape_incubation = (mean_incubation / sd_incubation) ** 2
scale_incubation = (sd_incubation ** 2) / mean_incubation

shape_generation = 2.39
scale_generation = 2.95

samples_sld = np.random.gamma(shape_sld, scale_sld, 10000)

samples_incubation = np.random.gamma(shape_incubation, scale_incubation, 10000)
combined_samples = samples_sld + samples_incubation
samples_generation = np.random.gamma(shape_generation, scale_generation, 10000)

x_values = np.linspace(0, 30, 3000)
sld_pdf = gamma.pdf(x_values, shape_sld, scale=scale_sld)
incubation_pdf = gamma.pdf(x_values, shape_incubation, scale=scale_incubation)
generation_pdf = gamma.pdf(x_values, shape_generation, scale=scale_generation)

In [ ]:

x = np.arange(len(omega_i))

plt.figure(figsize=(12, 8))

# Panel A
ax1 = plt.subplot(2, 3, 1)
ax1.bar(x, f_i, color='skyblue', edgecolor='black', alpha=0.7)
ax1.set_xlabel('Compartment number $i$', fontsize=12)
ax1.set_ylabel('Probability mass', fontsize=12)
ax1.tick_params(axis='both', labelsize=12)
ax1.set_ylim(0, 0.21)
ax1.text(-0.1, 1.05, 'A', transform=ax1.transAxes, fontsize=16, fontweight='bold')

# Panel D
ax4 = plt.subplot(2, 3, 4)
ax4.plot(t[:4000], fi_over_time[:4000], color='skyblue', marker='o', markersize=4, linestyle='-', alpha=0.7)
ax4.hist(samples_generation, bins=100, density=True, alpha=0.4, color='grey')
ax4.set_xlabel('Time (days)', fontsize=12)
ax4.set_ylabel('Probability density', fontsize=12)
ax4.tick_params(axis='both', labelsize=12)
#ax4.set_title('Infectivity Rate Over Time Since Infection', fontsize=11)
ax4.grid(True)
ax4.set_ylim(0, 0.16)
ax4.text(-0.1, 1.1, 'D', transform=ax4.transAxes, fontsize=16, fontweight='bold')

# Panel C
ax3 = plt.subplot(2, 3, 3)
ax3.bar(x, omega_i, color='purple', edgecolor='black', alpha=0.7)
ax3.set_xlabel('Compartment number $i$', fontsize=12)
ax3.set_ylabel('Probability mass', fontsize=12)
ax3.tick_params(axis='both', labelsize=12)
ax3.set_ylim(0, 0.21)
ax3.text(-0.1, 1.05, 'C', transform=ax3.transAxes, fontsize=16, fontweight='bold')

# Panel B
ax2 = plt.subplot(2, 3, 2)
ax2.bar(x, c_i, color='green', edgecolor='black', alpha=0.7)
ax2.set_xlabel('Compartment number $i$', fontsize=12)
ax2.set_ylabel('Probability mass', fontsize=12)
ax2.tick_params(axis='both', labelsize=12)
ax2.set_ylim(0, 0.21)
ax2.text(-0.1, 1.05, 'B', transform=ax2.transAxes, fontsize=16, fontweight='bold')

# Panel E
ax5 = plt.subplot(2, 3, 5)
ax5.plot(t[:4000], ci_over_time[:4000], color='green', marker='o', markersize=4, linestyle='-', alpha=0.7)
ax5.hist(samples_incubation, bins=100, density=True, alpha=0.4, color='grey')
ax5.set_xlabel('Time (days)', fontsize=12)
ax5.set_ylabel('Probability density', fontsize=12)
ax5.tick_params(axis='both', labelsize=12)
#ax5.set_title('Case Detection Rate Over Time Since Infection', fontsize=11)
ax5.grid(True)
ax5.set_ylim(0, 0.16)
ax5.text(-0.1, 1.1, 'E', transform=ax5.transAxes, fontsize=16, fontweight='bold')

# Panel F
ax6 = plt.subplot(2, 3, 6)
ax6.plot(t[:4000], shedding_concentration_over_time[:4000], color='purple', marker='o', markersize=4, linestyle='-', alpha=0.7)
ax6.hist(combined_samples, bins=100, density=True, alpha=0.4, color='grey')
ax6.set_xlabel('Time (days)', fontsize=12)
ax6.set_ylabel('Probability density', fontsize=12)
ax6.tick_params(axis='both', labelsize=12)
#ax6.set_title('Shedding Rate Over Time Since Infection', fontsize=11)
ax6.grid(True)
ax6.set_ylim(0, 0.16)
ax6.set_xlim(0, 40)
ax6.text(-0.1, 1.1, 'F', transform=ax6.transAxes, fontsize=16, fontweight='bold')

plt.tight_layout()
plt.show()

### Figure 2

In [ ]:
Re_list = pd.read_csv("data/mock dataset/Re_list.csv", header=None)
Re_list = Re_list[0].to_numpy()
Ivector = pd.read_csv("data/mock dataset/Ivector.csv", header=None)
Ivector = Ivector.to_numpy()
Scvector = pd.read_csv("data/mock dataset/Scvector.csv", header=None)
Scvector = Scvector[0].to_numpy()
Swvector = pd.read_csv("data/mock dataset/Swvector.csv", header=None)
Swvector = Swvector[0].to_numpy()
tvector = pd.read_csv("data/mock dataset/tvector.csv", header=None)
tvector = tvector[0].to_numpy()
t_ww = pd.read_csv("data/mock dataset/t_ww.csv", header=None)
t_ww = t_ww[0].to_numpy()
cases = pd.read_csv("data/mock dataset/cases.csv", header=None)
cases = cases[0].to_numpy()
t_case = pd.read_csv("data/mock dataset//t_cases.csv", header=None)
t_case = t_case[0].to_numpy()
ww = pd.read_csv("data/mock dataset//ww.csv", header=None)
ww = ww[0].to_numpy()

In [ ]:
start, end = 13880, 21190
mask_c = (t_case >= start/10) & (t_case <= end/10)
mask_w = (t_ww >= start/10) & (t_ww <= end/10)
cases_in_range = cases[mask_c]
t_case_in_range = t_case[mask_c]
ww_in_range = ww[mask_w]
t_ww_in_range = t_ww[mask_w]
weekly_incidence = util.calculate_weekly_incidence(Scvector[start:end])
sub_t  = tvector[start:end]
weekly_time = sub_t[70::70]
Ivector_mock = Ivector[13880:21190,:]

In [ ]:
plt.figure(figsize=(15, 10))
label_font = {'fontsize': 15}

# --- Plot 1: Time-Varying Effective Reproduction Number ---
ax1 = plt.subplot(2, 2, 1)
ax1.plot(tvector[start:end] / 365, Re_list[start:end],
         linestyle='-', label='Effective reproduction number', color='black')
ax1.set_xlabel('Year', **label_font)
ax1.set_ylabel('Effective reproduction number', **label_font)
ax1.grid(True)
ax1.text(0.97, 1.05,'A', transform=ax1.transAxes,
         fontsize=16, fontweight='bold')

# --- Plot 2: Infection Dynamics ---
n_time, k = Ivector_mock.shape  # Number of time points and compartments
cmap = plt.get_cmap("gray")
colors = [cmap(i / (k - 1)) for i in range(k)]
ax2 = plt.subplot(2, 2, 2)
for i in reversed(range(k)):
    ax2.plot(tvector[start:end] / 365, Ivector_mock[:, i],
             marker='o', linestyle='-', color=colors[i])
ax2.set_ylim(0, 350)
ax2.set_xlabel('Year', **label_font)
ax2.set_ylabel("Number of individuals", **label_font)
ax2.grid(True)
ax2.text(0.97, 1.05,'B', transform=ax2.transAxes,
         fontsize=16, fontweight='bold')

# --- Plot 3: Weekly Case Incidence with Noisy measured data ---
ax3 = plt.subplot(2, 2, 3)
ax3.bar(weekly_time / 365, weekly_incidence,
        color='black', alpha=0.45, width=0.03, label='Weekly case incidence')
ax3.scatter(t_case_in_range / 365, cases_in_range,
            marker='o', linestyle='-', color='red', label='Noisy measured data')
ax3.set_xlabel("Year", **label_font)
ax3.set_ylabel("Weekly case incidence", **label_font)
ax3.grid(True)
ax3.text(0.97, 1.05, 'C', transform=ax3.transAxes,
         fontsize=16, fontweight='bold')

# --- Plot 4: Wastewater Viral Concentrations with Noisy measured data ---
ax4 = plt.subplot(2, 2, 4)
ax4.scatter(t_ww_in_range / 365, ww_in_range, linestyle='-', color='red', label='Noisy measured data')
ax4.plot(tvector[start:end] / 365, Swvector[start:end],
         linestyle='-', color='black', label='Wastewater virus concentration')
ax4.set_xlabel("Year", **label_font)
ax4.set_ylabel("Wastewater virus concentration", **label_font)
ax4.grid(True)
ax4.text(0.97, 1.05, 'D', transform=ax4.transAxes,
         fontsize=16, fontweight='bold')

# --- Adjust x-axis tick labels for all subplots ---
for ax in plt.gcf().get_axes():
    ax.xaxis.set_major_locator(ticker.MultipleLocator(1))
    for label in ax.get_xticklabels():
        label.set_fontsize(12)
    for label in ax.get_yticklabels():
        label.set_fontsize(12)

plt.tight_layout()
plt.show()


### Figure 3

In [ ]:
ZH_ww = pd.read_csv('data/ZH_WW_data.csv')
ZH_case = pd.read_csv('data/ZH_case_incidence_data.csv')
ZH_ww['date'] = pd.to_datetime(ZH_ww['date'])
ZH_case['date'] = pd.to_datetime(ZH_case['date'])

highlight_dates = ['2021-01-10', '2020-12-13', '2021-01-12']
highlight_dates_dt = pd.to_datetime(highlight_dates)

ZH_ww_excluded = ZH_ww[~ZH_ww['date'].isin(highlight_dates_dt)]

start_date = pd.to_datetime('2020-09-03')
end_date = pd.to_datetime('2021-01-19')

ZH_ww_selected = ZH_ww_excluded[
    (ZH_ww_excluded['date'] >= start_date) & (ZH_ww_excluded['date'] <= end_date)
]
ZH_case_selected = ZH_case[
    (ZH_case['date'] >= start_date) & (ZH_case['date'] <= end_date)
]

ZH_ww_excluded_points = ZH_ww[
    (ZH_ww['date'] >= start_date) &
    (ZH_ww['date'] <= end_date) &
    (ZH_ww['date'].isin(highlight_dates_dt))
]


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(9, 4))

# --- Cases panel ---
axs[0].plot(
    ZH_case_selected['date'],
    ZH_case_selected['confirmed'],
    marker='o',
    linestyle='-',
    label='Confirmed Cases',
    color = "black",
    linewidth=1.5
)

# Force one tick per month
axs[0].xaxis.set_major_locator(mdates.MonthLocator())
axs[0].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d/%Y'))
axs[0].tick_params(axis='x', rotation=45)
axs[0].set_ylabel('Confirmed Cases')


# --- Wastewater panel ---
line_n1, = axs[1].plot(
    ZH_ww_selected['date'],
    ZH_ww_selected['n1'],
    label='n1',
    marker='o',
    linestyle='-'
)
line_n2, = axs[1].plot(
    ZH_ww_selected['date'],
    ZH_ww_selected['n2'],
    label='n2',
    marker='o',
    linestyle='-'
)

color_n1 = line_n1.get_color()
color_n2 = line_n2.get_color()
axs[0].text(0.95, 1.05, 'A', transform=axs[0].transAxes,
        fontsize=14, fontweight='bold')


axs[1].scatter(
    ZH_ww_excluded_points['date'],
    ZH_ww_excluded_points['n1'],
    marker='x',
    color=color_n1,
    label='n1 (excluded)'
)
axs[1].scatter(
    ZH_ww_excluded_points['date'],
    ZH_ww_excluded_points['n2'],
    marker='x',
    color=color_n2,
    label='n2 (excluded)'
)

# Force one tick per month
axs[1].text(2.15, 1.05, 'B', transform=axs[0].transAxes,
        fontsize=14, fontweight='bold')
axs[1].xaxis.set_major_locator(mdates.MonthLocator())
axs[1].xaxis.set_major_formatter(mdates.DateFormatter('%m/%d/%Y'))
axs[1].tick_params(axis='x', rotation=45)
axs[1].set_ylabel('Virus concentration (gene copies/mL)')
plt.tight_layout()
plt.show()
